In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from scipy import stats
from datetime import datetime, timedelta
from pytz import timezone, UTC
from typing import Union, List
import matplotlib.pyplot as plt
from matplotlib.pyplot import text
csfont = {'fontname':'Times'}
from cycler import cycler

In [ ]:
precision = 1  # Precision of 1 minute

# We focus on the time period between 10 am and 4 pm
start_hour = 10 
end_hour = 16

time_focus_start = start_hour * 60 // precision  # 10 AM
time_focus_end = end_hour * 60 // precision   # 4 PM

# Total length of the time horizon in 1 minutes
length_time = int(time_focus_end - time_focus_start)

# Time constants
MINUTES_IN_HOUR = 60
SECONDS_IN_HOUR = 3600

# Path to the folder including the csv files
data_folder_path = '/Users/ebrukasikaralar/data_parallel_server/data_july02_oct02/csv_files/'

In [ ]:
def get_file_list(folder_path, file_pattern):
    """
    Generates a list of file paths from a specified folder that match a given pattern.

    Parameters:
    - folder_path: str - Path to the folder where files are located.
    - file_pattern: str - Pattern to match files.

    Returns:
    - list: List of file paths matching the pattern in the specified folder.
    """
    
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"The specified folder path does not exist: {folder_path}")

    file_list = glob.glob(os.path.join(folder_path, file_pattern))

    if not file_list:
        print(f"No files found in {folder_path} matching the pattern '{file_pattern}'")

    return file_list

In [ ]:
file_list_cust_subcalls = get_file_list(data_folder_path, file_pattern="*_cust_subcalls.csv")

In [ ]:
all_nodes = set()

for file in file_list_cust_subcalls:
    data = pd.read_csv(file)
    service1_nodes = data["node"].unique()
    all_nodes.update(service1_nodes)

print("Unique nodes for service 1 across all files:", sorted(all_nodes))

In [ ]:
def filt_arrivals(file_path, service, node = 0):
    """
    Filters and processes call arrival data for a given service (and node if service=1).

    Parameters:
    - file_path: str - Path to the data file.
    - service: int - Service type to filter.
    - node: int - Node number to filter (only applies when service == 1).

    Returns:
    - float: Average hourly arrival rate between 10 AM and 4 PM for the file (day).
    """ 
    try:
        data = pd.read_csv(file_path)
    except Exception as e:
        raise FileNotFoundError(f"Error reading file {file_path}: {e}")
        
    
    # Filter based on queue time and service time
    data = data[(data["queue_time"] < 900) & (data["service_time"] < 1800)]

    
    # Remove calls with abnormal outcomes
    abnormal_outcomes = [4,13,14,23,30,40,50]
    data = data[~data["call_id"].isin(data[data["outcome"].isin(abnormal_outcomes)]["call_id"].unique())]

    
    # Focus on first customer subcalls and remove duplicates
    data = data[data["cust_subcall"] == 1].sort_values(["segment_start"]).drop_duplicates("call_id", keep="last")
    
    # Filter data based on service and node
    data = data[data["service"] == service]
        
    if (service == 1) and (node != 0):
        data = data[data["node"].isin([node])]
    
    # Convert segment_start to datetime and adjust for timezone
    data["Datetime"] = pd.to_datetime(data["segment_start"], unit="s", utc=True)
    data["day"] = data["Datetime"].dt.date

    new_data = data
    drop_ind = new_data.drop_duplicates(subset = ["day"])
    
    if len(drop_ind) > 1:
        drop_day = np.array(drop_ind.day)[1]
        data = data[data["day"] != drop_day]
    
    # Filter data between 10 AM and 4 PM
    data = data.set_index('Datetime')
    filtered_data = data.between_time("10:00", "16:00")
    return len(filtered_data)/(end_hour - start_hour)

In [ ]:
def average_rate(file_list, service, node):
    """
    Computes the average hourly arrival rate between 10 AM and 4 PM across multiple days.

    Parameters:
    - file_list: list of str - List of file paths (one per day).
    - service: int - Service type to filter.
    - node: int - Node number to filter (only used when service == 1).

    Returns:
    - float: Average hourly arrival rate across all files (days).
    """
    
    if not file_list:
        raise ValueError("The file list is empty.")
    
    total_hourly_rates = 0.0
   
    for file in file_list:
        total_hourly_rates += filt_arrivals(file, service, node)

    return total_hourly_rates / len(file_list)

In [ ]:
arrival_rate_records = []

classes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
nodes = [1, 2, 3, 7]  # Only for class 1 (Retail)

for cls in classes:
    if cls == 1:
        for node in nodes:
            rate = average_rate(file_list_cust_subcalls, cls, node=node)
            print(cls, node, rate)
            arrival_rate_records.append({"class": cls, "node": node, "hourly rate": rate})
    else:
        rate = average_rate(file_list_cust_subcalls, cls, node=0)
        print(cls, 0, rate)
        arrival_rate_records.append({"class": cls, "node": 0, "hourly rate": rate})

# Convert to DataFrame at the end
arrival_rate_df = pd.DataFrame(arrival_rate_records)

In [ ]:
def filt_arrivals_precision(file_path,precision,service_lookup,service,node=0):
    """
    Filter and process call arrival data.

    Parameters:
    - file_path: str - Path to the data file.
    - precision: int - Time precision for resampling.
    - service_lookup: bool - Whether to filter by service.
    - service: int - Service type to filter.
    - node: int - Node number to filter, default is 0.

    Returns:
    - DataFrame: Processed data with columns ['index', 'call_id'].
    """
    
    try:
        data = pd.read_csv(file_path)
    except Exception as e:
        raise FileNotFoundError(f"Error reading file {file_path}: {e}")
        
    
    # Filter based on queue time and service time
    data = data[(data["queue_time"] < 900) & (data["service_time"] < 1800)]
 
    # Remove calls with abnormal outcomes
    abnormal_outcomes = [4,13,14,23,30,40,50]
    calls_with_abnormal_outcome = data[data["outcome"].isin(abnormal_outcomes)]
    call_ids_abnormal_outcome = calls_with_abnormal_outcome["call_id"].unique()
    drop_index = data[data["call_id"].isin(call_ids_abnormal_outcome)].index
    data = data.drop(index = drop_index)
    
    # Focusing only on the first customer subcalls
    data = data[data["cust_subcall"] == 1]
    
    # Removing the multiple records of the same call
    data = data.sort_values(["segment_start"])
    data = data.drop_duplicates("call_id",keep = "last")
      
    # Filter data based on service and node
    if service_lookup == True:
        data = data[data["service"] == service]
        
        if (service == 1) and (node != 0):
            data = data[data["node"].isin([node])]     
    
    # Convert segment_start to datetime and adjust for timezone
    data["Datetime"] = pd.to_datetime(data["segment_start"], unit="s", utc=True)
    data["day"] = data["Datetime"].dt.date

    new_data = data
    drop_ind = new_data.drop_duplicates(subset = ["day"])
    
    if len(drop_ind) > 1:
        drop_day = np.array(drop_ind.day)[1]
        data = data[data["day"] != drop_day]
    
    data = data.set_index('Datetime')
    
    # Resolution
    resolution = "{pre}T"
    
    # How many arrivals happened within the specific time interval
    data = data.resample(resolution.format(pre = precision)).count()
    data = data.dropna(axis = 0)
    
    # Setting up a new column to show the time interval
    new_index = []
    for j in data.index:
        h=j.hour
        m=j.minute
        item=str(h)+':'+str(m)
        new_index.append(item)
        
    data["index"] = new_index
    data = data.reset_index()
    
    return data[["index","call_id"]]

In [ ]:
def format_time(hour, minute):
    """Formats time as 'hour:minute'."""
    return f"{hour}:{minute:0d}"
    
def create_time_intervals():
    """Generates a list of time intervals from '0:0' to '23:59' in 1-minute increments."""
    return [format_time(hour, minute) for hour in range(24) for minute in range(0, 60, precision)]

def merge_daily_data(file_list, precision, service_lookup, service, node):
    """
    Merges and processes data from multiple days.

    Parameters:
    - file_list: list of str - List of file paths.
    - precision: int - Time precision for resampling.
    - service_lookup: bool - Whether to filter by service.
    - service: int - Service type to filter.
    - node: int - Node number to filter.

    Returns:
    - DataFrame: Merged and processed data.
    """
    
    if not file_list:
        
        raise ValueError("The file list is empty.")
        
    time_intervals = create_time_intervals()
    main_dataframe = pd.DataFrame(index=pd.Index(time_intervals))
    main_dataframe = main_dataframe.merge(filt_arrivals_precision(file_list[0],precision,service_lookup,service,node), how = "left", left_on = main_dataframe.index, right_on = "index", suffixes=('', '_0'))
    main_dataframe = main_dataframe.fillna(0)
    
    for i in range(1,len(file_list)):
        
        # Merging all the days based on their time arrival 
        main_dataframe = main_dataframe.merge(filt_arrivals_precision(file_list[i],precision,service_lookup,service,node), how = "left", left_on = "index", right_on = "index", suffixes=('', f'_{i}'))
    
    # Dropping the null values
    main_dataframe = main_dataframe.fillna(0)
    
    # Setting the index to show the arrival intervals
    main_dataframe = main_dataframe.set_index("index")
    
    
    return main_dataframe
        

def means_totals(file_list, precision, service_lookup, service, node=0):
    """
    Calculates the average arrival rate over multiple days.

    Parameters:
    - file_list: list of str - List of file paths.
    - precision: int - Time precision for resampling.
    - service_lookup: bool - Whether to filter by service.
    - service: int - Service type to filter.
    - node: int - Node number to filter, default is 0.

    Returns:
    - Series: Average arrival rate for each time interval.
    """
    merged_data = merge_daily_data(file_list, precision, service_lookup, service, node)
    
    mean = merged_data.apply(np.mean, axis=1)
    var = merged_data.apply(np.var, axis=1)
    
    return mean, var

In [ ]:
def process_class_data(class_index, file_list, precision, nodes=None):
    """
    Processes data for a given class index and nodes.

    Parameters:
    - class_index: int - The class index to process.
    - file_list: list - List of file paths for data processing.
    - precision: int - Time precision for resampling.
    - nodes: list or None - List of node numbers to process for the class. If None, process without nodes.

    Returns:
    - dict: Dictionary of the processed data.
    """
    results = {}
    if nodes:
        for node in nodes:
            key = f'class_{class_index}_node_{node}'
            results[key] = means_totals(file_list, precision, service_lookup=True, service=class_index, node=node)
    else:
        key = f'class_{class_index}'
        results[key] = means_totals(file_list, precision, service_lookup=True, service=class_index, node=0)

    return results

In [ ]:
tot_mean, tot_var = means_totals(file_list_cust_subcalls, precision=1, service_lookup=False, service=0, node=0)

In [ ]:
import scipy.stats as stats
lower_bounds = np.zeros((24*MINUTES_IN_HOUR,))
upper_bounds = np.zeros((24*MINUTES_IN_HOUR,))

def confidence_interval(sample_mean, sample_variance, sample_size):
    confidence_level = 0.95
    z_score = stats.norm.ppf(1 - (1 - confidence_level) / 2)
    standard_error = (sample_variance / sample_size) ** 0.5
    margin_of_error = z_score * standard_error
    lower_bound = sample_mean - margin_of_error
    upper_bound = sample_mean + margin_of_error
    return lower_bound, upper_bound


for i in range(0,24*60):
    sample_mean = tot_mean[i]
    sample_variance = tot_var[i]
    sample_size = len(file_list_cust_subcalls)
    lower_bound, upper_bound = confidence_interval(sample_mean, sample_variance, sample_size)
    lower_bounds[i] = lower_bound
    upper_bounds[i] = upper_bound

In [ ]:
total_mean = np.array(tot_mean)

In [ ]:
output_index = np.arange(0,24*MINUTES_IN_HOUR,1)
plt.step(output_index, total_mean*MINUTES_IN_HOUR,   color= "black")
plt.plot(output_index, lower_bounds*MINUTES_IN_HOUR, color="black", alpha = 0.45, linestyle = "dashed")
plt.plot(output_index, upper_bounds*MINUTES_IN_HOUR, color="black", alpha = 0.45, linestyle = "dashed")
x_ticks = np.arange(0,1441,180)
y_ticks = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000]

plt.yticks(y_ticks, **csfont, size = 13)
plt.xticks(x_ticks, ["0:00", "3:00","6:00","9:00","12:00","15:00","18:00","21:00","24:00"],**csfont,size=13)
plt.vlines(10*MINUTES_IN_HOUR, 0, 7000, color='black', label = "9 A.M.",linestyle = "dashed")
plt.vlines(16*MINUTES_IN_HOUR, 0, 7000, color='black', label = "5 P.M.",linestyle = "dashed")

plt.ylabel("Number of Arrivals per hour",**csfont,size =15)
plt.xlabel("Time of the day",**csfont, size = 15)
plt.gcf().set_size_inches(12,4)
plt.ylim([0,7000])

text(8*MINUTES_IN_HOUR, 6550, "10 A.M.", rotation=360, verticalalignment='center',**csfont, size=13)
text(14.5*MINUTES_IN_HOUR, 6550, "4 P.M.", rotation=360, verticalalignment='center',**csfont, size=13)
plt.savefig("arrivals_CI.png")

In [ ]:
def save_data_to_csv(data, filename, time_slice=None):
    """
    Saves given data to a CSV file.

    Parameters:
    - data: DataFrame or Series - The data to be saved.
    - filename: str - The name of the file to save the data.
    - time_slice: tuple of int, optional - Start and end indices for slicing the data by time. If None, save all data.
    """
    if time_slice:
        sliced_data = data[time_slice[0]:time_slice[1]]
    else:
        sliced_data = data

    try:
        np.savetxt(filename, sliced_data, delimiter=",")
    except Exception as e:
        print(f"Error saving data to {filename}: {e}")

In [ ]:
class_indices = [1, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

# Results are stored in results_by_class
results_by_class = {}

for class_ind in class_indices:
    if class_ind == 1:  # class 1 is the retail class
        results_by_class.update(process_class_data(class_ind, file_list_cust_subcalls, precision, nodes=[1, 2, 3]))
    else:
        results_by_class.update(process_class_data(class_ind, file_list_cust_subcalls, precision))

In [ ]:
class_1_node_1_arrivals = results_by_class["class_1_node_1"]
class_1_node_2_arrivals = results_by_class["class_1_node_2"]
class_1_node_3_arrivals = results_by_class["class_1_node_3"]
class_2_arrivals = results_by_class["class_2"]
class_3_arrivals = results_by_class["class_3"]
class_4_arrivals = results_by_class["class_4"]
class_5_arrivals = results_by_class["class_5"]
class_6_arrivals = results_by_class["class_6"]
class_7_arrivals = results_by_class["class_7"]
class_8_arrivals = results_by_class["class_8"]
class_9_arrivals = results_by_class["class_9"]
class_10_arrivals = results_by_class["class_10"]
class_11_arrivals = results_by_class["class_11"]

In [ ]:
for class_ind in class_indices:
    if class_ind == 1:  # Assuming class 1 is the retail class
        # Retail class is divided into three nodes 
        for node_num in [1, 2, 3]:
            file_name_all = f"main_test_arrivals_class{class_ind}_node{node_num}_all_{precision}min.csv"
            file_name_partial = f"main_test_arrivals_class{class_ind}_node{node_num}_partial_{precision}min.csv"
            save_data_to_csv(results_by_class[f"class_{class_ind}_node_{node_num}"], file_name_all)
            save_data_to_csv(results_by_class[f"class_{class_ind}_node_{node_num}"], file_name_partial, time_slice=(time_focus_start, time_focus_end))
    
    # remaining classes other than Retail class
    else:
        file_name_all = f"main_test_arrivals_class{class_ind}_all_{precision}min.csv"
        file_name_partial = f"main_test_arrivals_class{class_ind}_partial_{precision}min.csv"
        save_data_to_csv(results_by_class[f"class_{class_ind}"], file_name_all)
        save_data_to_csv(results_by_class[f"class_{class_ind}"], file_name_partial, time_slice=(time_focus_start, time_focus_end))